# 08 · Two-Pass Saliency Inference

Implements a **saliency-guided anomaly detection** pipeline on top of the trained
`diffusion_best.pt` / `diffusion_epoch0030.pt` checkpoint.

```
BraTS T1 slice  x₀
      │
      ├──── PASS 1: full-image DDPM ────────────────────────────────────────┐
      │     add noise to t=T_enc → DDPM reverse → x̂₀_rough               │
      │     anomaly_rough = |x₀ − x̂₀_rough|                               │
      │     saliency_mask = threshold(anomaly_rough)   ◄───────────────────┘
      │
      └──── PASS 2: hybrid DDIM/DDPM ─────────────────────────────────────┐
            add noise to t=T_enc → hybrid_sample() → x̂₀_clean            │
              • DDPM (stochastic) inside saliency_mask                     │
              • DDIM (deterministic) outside saliency_mask                 │
            anomaly_clean = |x₀ − x̂₀_clean|                               │
            ◄───────────────────────────────────────────────────────────────┘
```

**Flag**: set `USE_SALIENCY = False` to fall back to the original single-pass pipeline.

**Sections**
1. Imports & config
2. Data loading helpers (T1 + seg preprocessing)
3. Model definition & checkpoint loading
4. Schedulers
5. Pass 1 — full-image DDPM inference
6. Thresholding — Otsu vs fixed
7. `hybrid_sample` — per-pixel DDPM/DDIM blending
8. Pass 2 — saliency-guided hybrid inference
9. Full two-pass pipeline wrapper
10. Single-slice demo with visualisation
11. Batch evaluation (DSC, AUROC, pixel-AP)

## 1 · Imports & config

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter
from skimage.filters import threshold_otsu
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset

from monai.networks.nets import DiffusionModelUNet
from monai.networks.schedulers import DDIMScheduler, DDPMScheduler

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ── device ───────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.bfloat16   # matches training; change to float32 if no bfloat16 support
print(f'Device : {DEVICE}  |  dtype : {DTYPE}')

# ── paths ─────────────────────────────────────────────────────────────────────
def _find_dir(name):
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / name
        if p.exists():
            return p
    raise FileNotFoundError(f'{name!r} not found from {Path.cwd()}')

BRATS_ROOT = _find_dir('BraTS2020_TrainingData')
# prefer diffusion_best.pt; fall back to the epoch checkpoint
for _name in ['diffusion_best.pt', 'diffusion_epoch0030.pt']:
    _p = _find_dir('.').parent / _name if not (_find_dir('.')).parent.exists() else None
    # search up from cwd
    for _base in [Path.cwd(), *Path.cwd().parents]:
        _candidate = _base / _name
        if _candidate.exists():
            CKPT_PATH = _candidate
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('No checkpoint found. Expected diffusion_best.pt or diffusion_epoch0030.pt')

print(f'BraTS  : {BRATS_ROOT}')
print(f'Ckpt   : {CKPT_PATH}')

# ── pipeline flags ────────────────────────────────────────────────────────────
USE_SALIENCY   = True    # False → original single-pass DDPM pipeline
T_ENCODE       = 500     # noise level for both passes  (try 250 / 500 / 750)
N_DDPM_STEPS   = 500     # reverse steps for Pass 1 (full DDPM; uses every step by default)
N_DDIM_STEPS   = 50      # reverse steps for Pass 2 DDIM portion
SMOOTH_SIGMA   = 2.0     # Gaussian smoothing on anomaly maps (pixels)
THRESH_METHOD  = 'otsu'  # 'otsu' or 'fixed'
FIXED_THRESH   = 0.10    # used only when THRESH_METHOD == 'fixed'

## 2 · Data loading helpers

In [ ]:
def preprocess_volume(nii_path, target_size=256, z_low=0.15, z_high=0.85, fill_thresh=0.05):
    """Preprocess a T1w NIfTI volume → list of (1, H, W) float32 slices in [0,1].
    Identical to the training pipeline in 06_train_from_scratch."""
    vol = nib.load(str(nii_path)).get_fdata(dtype=np.float32)
    if vol.ndim != 3:
        return [], []
    nonzero = vol[vol != 0]
    if nonzero.size == 0:
        return [], []
    p_lo, p_hi = np.percentile(nonzero, [0.5, 99.5])
    vol = np.clip((vol - p_lo) / max(float(p_hi - p_lo), 1e-6), 0.0, 1.0)
    D   = vol.shape[2]
    z0, z1 = int(np.floor(z_low * D)), int(np.ceil(z_high * D))
    slices, z_indices = [], []
    for z in range(z0, z1):
        sl = vol[:, :, z]
        if sl.mean() < fill_thresh:
            continue
        sl_t = torch.from_numpy(sl)[None, None]          # (1,1,H,W)
        sl_r = F.interpolate(sl_t, size=(target_size, target_size),
                             mode='bilinear', align_corners=False)
        slices.append(sl_r[0].numpy())                   # (1,H,W)
        z_indices.append(z)
    return slices, z_indices


def preprocess_seg(seg_path, z_indices, target_size=256):
    """Preprocess a BraTS segmentation volume, keeping only the z-indices that
    survived the T1 fill filter.  Returns list of (1, H, W) int32 arrays."""
    seg = nib.load(str(seg_path)).get_fdata(dtype=np.float32).astype(np.int32)
    out = []
    for z in z_indices:
        sl = seg[:, :, z].astype(np.float32)
        sl_t = torch.from_numpy(sl)[None, None]
        sl_r = F.interpolate(sl_t, size=(target_size, target_size),
                             mode='nearest')
        out.append(sl_r[0].numpy().astype(np.int32))     # (1,H,W)
    return out


def discover_brats_cases(brats_root):
    cases = []
    for case_dir in sorted(Path(brats_root).iterdir()):
        if not case_dir.is_dir():
            continue
        t1_files  = list(case_dir.glob('*_t1.nii'))
        seg_files = list(case_dir.glob('*_seg.nii'))
        if t1_files and seg_files:
            cases.append(dict(id=case_dir.name,
                              t1_path=t1_files[0],
                              seg_path=seg_files[0]))
    return cases


class BraTSDataset(Dataset):
    """Flat dataset: one item per valid axial slice."""
    def __init__(self, cases, tumor_only=False):
        self.items = []
        for c in cases:
            slices, z_ids = preprocess_volume(c['t1_path'])
            segs           = preprocess_seg(c['seg_path'], z_ids)
            for s, z, seg in zip(slices, z_ids, segs):
                has_tumor = bool(seg.max() > 0)
                if tumor_only and not has_tumor:
                    continue
                self.items.append(dict(
                    image    =torch.from_numpy(s),          # (1,256,256)
                    seg      =seg,                           # (1,256,256) int32
                    case     =c['id'],
                    z_idx    =z,
                    has_tumor=has_tumor,
                ))

    def __len__(self):  return len(self.items)
    def __getitem__(self, i): return self.items[i]


def brats_collate(batch):
    return dict(
        image    =torch.stack([b['image'] for b in batch]),
        seg      =np.stack([b['seg'] for b in batch]),
        case     =[b['case'] for b in batch],
        z_idx    =[b['z_idx'] for b in batch],
        has_tumor=[b['has_tumor'] for b in batch],
    )


# ── discover cases ────────────────────────────────────────────────────────────
all_cases = discover_brats_cases(BRATS_ROOT)
print(f'Found {len(all_cases)} BraTS cases')

## 3 · Model definition & checkpoint loading

In [ ]:
# Architecture must match the saved checkpoint exactly.
# diffusion_epoch0030.pt was trained with:
#   channels=(64,128,256,256), attention_levels=(False,False,False,True),
#   num_res_blocks=1, out_channels=1
unet = DiffusionModelUNet(
    spatial_dims     = 2,
    in_channels      = 1,
    out_channels     = 1,
    channels         = (64, 128, 256, 256),
    attention_levels = (False, False, False, True),
    num_res_blocks   = 1,
    num_head_channels= 32,
    norm_num_groups  = 32,
).to(DEVICE)

total_params = sum(p.numel() for p in unet.parameters())
print(f'Model: {total_params/1e6:.2f} M parameters')

# ── load checkpoint ───────────────────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)

# support both 'unet' and 'unet_full' key names
state_key = 'unet' if 'unet' in ckpt else 'unet_full'
missing, unexpected = unet.load_state_dict(ckpt[state_key], strict=True)
if missing:     print(f'WARNING missing   : {missing[:5]}')
if unexpected:  print(f'WARNING unexpected: {unexpected[:5]}')

epoch_loaded = ckpt.get('epoch', '?')
print(f'Loaded epoch {epoch_loaded} from {CKPT_PATH.name}')

# cast to bfloat16 (matches training)
unet = unet.to(DTYPE).eval()
print(f'Model dtype: {next(unet.parameters()).dtype}')

## 4 · Schedulers

In [ ]:
T = 1000
_sched_kwargs = dict(
    num_train_timesteps = T,
    schedule            = 'linear_beta',
    beta_start          = 1e-4,
    beta_end            = 0.02,
    clip_sample         = False,
)

ddpm_scheduler = DDPMScheduler(**_sched_kwargs)
ddim_scheduler = DDIMScheduler(**_sched_kwargs)

print(f'DDPM scheduler ready  (T={T})')
print(f'DDIM scheduler ready  (T={T})')

## 5 · Pass 1 — full-image DDPM inference

The original pipeline: noise the whole image to `T_ENCODE`, reverse with DDPM
(every step), compute `|x₀ − x̂₀|` as the rough anomaly map.

In [ ]:
@torch.no_grad()
def ddpm_reconstruct(x0: torch.Tensor,
                     encode_t: int = T_ENCODE) -> torch.Tensor:
    """
    Full-image DDPM reconstruction (Pass 1 / fallback pipeline).

    Parameters
    ----------
    x0       : (1,1,256,256) float32 on DEVICE, values in [0,1]
    encode_t : noise level — higher = more corruption, stronger anomaly signal

    Returns
    -------
    x_hat : (1,1,256,256) float32  healthy reconstruction
    """
    # scale to [-1, 1] for the model
    x0_scaled = x0 * 2.0 - 1.0

    # forward: add noise to encode_t
    noise = torch.randn_like(x0_scaled)
    t_vec = torch.tensor([encode_t], device=DEVICE).long()
    x_t   = ddpm_scheduler.add_noise(x0_scaled, noise, t_vec)

    # reverse: DDPM from encode_t → 0  (iterate over every integer step)
    # DDPMScheduler expects to start from timestep T-1 but we only need encode_t→0
    timesteps = list(range(encode_t, -1, -1))
    x_cur = x_t.to(DTYPE)
    for t_val in timesteps:
        t_batch    = torch.full((x_cur.shape[0],), t_val, device=DEVICE, dtype=torch.long)
        noise_pred = unet(x_cur, t_batch)                 # (1,1,256,256)
        x_cur      = ddpm_scheduler.step(noise_pred.float(), t_val, x_cur.float())[0].to(DTYPE)

    # scale back to [0,1]
    x_hat = ((x_cur.float() + 1.0) / 2.0).clamp(0.0, 1.0)
    return x_hat


def anomaly_map(x0: torch.Tensor,
                x_hat: torch.Tensor,
                smooth_sigma: float = SMOOTH_SIGMA) -> np.ndarray:
    """
    Pixel-wise |x₀ − x̂₀| with optional Gaussian smoothing.

    Returns (256, 256) float32 numpy array.
    """
    diff = (x0.float() - x_hat.float()).abs()[0, 0].cpu().numpy()
    if smooth_sigma > 0:
        diff = gaussian_filter(diff, sigma=smooth_sigma)
    return diff.astype(np.float32)

## 6 · Thresholding — Otsu vs fixed

The saliency mask for Pass 2 is derived by thresholding the Pass 1 anomaly map.
Two strategies are compared:

| Method | When to use |
|--------|-------------|
| **Otsu** | Adaptive; works well when anomaly pixels are genuinely bimodal. May over-segment on near-uniform maps. |
| **Fixed** | Use when you want reproducible behaviour; tune on a validation set. A value of ~0.05–0.15 typically works for T_ENCODE=500. |

In [ ]:
def build_saliency_mask(amap: np.ndarray,
                        method: str   = THRESH_METHOD,
                        fixed_val: float = FIXED_THRESH,
                        min_area: int = 50) -> torch.Tensor:
    """
    Threshold the anomaly map to produce a binary saliency mask.

    Parameters
    ----------
    amap      : (H, W) float32 anomaly map
    method    : 'otsu' | 'fixed'
    fixed_val : threshold value when method='fixed'
    min_area  : masks with fewer positive pixels than this fall back to zeros
                (avoids masking the whole image on near-uniform maps)

    Returns
    -------
    mask : (1,1,H,W) float32 tensor on DEVICE, values in {0, 1}
    """
    if method == 'otsu':
        try:
            thresh = threshold_otsu(amap)
        except Exception:
            thresh = fixed_val  # fallback if otsu fails (e.g. flat map)
    elif method == 'fixed':
        thresh = fixed_val
    else:
        raise ValueError(f'Unknown threshold method: {method!r}')

    binary = (amap >= thresh).astype(np.float32)   # (H, W)

    if binary.sum() < min_area:
        binary = np.zeros_like(binary)

    mask_t = torch.from_numpy(binary)[None, None].to(DEVICE)   # (1,1,H,W)
    return mask_t, float(thresh)


def compare_thresholds(amap: np.ndarray, gt_mask: np.ndarray = None):
    """
    Print a comparison table of Otsu vs a range of fixed thresholds.
    Optionally computes DSC against a ground-truth binary mask.
    """
    otsu_t = threshold_otsu(amap)
    candidates = sorted(set([otsu_t] + list(np.arange(0.02, 0.25, 0.02))))

    print(f'{'Threshold':>12}  {'% pos':>7}  {'Method':>8}', end='')
    if gt_mask is not None:
        print(f'  {'DSC':>6}', end='')
    print()
    print('-' * (35 + (8 if gt_mask is not None else 0)))

    for t in candidates:
        binary = (amap >= t).astype(np.float32)
        pct    = 100.0 * binary.mean()
        label  = 'OTSU' if abs(t - otsu_t) < 1e-6 else 'fixed'
        row    = f'{t:>12.4f}  {pct:>7.2f}%  {label:>8}'
        if gt_mask is not None:
            gt_b = (gt_mask > 0).astype(np.float32)
            inter = (binary * gt_b).sum()
            dsc   = 2 * inter / max(binary.sum() + gt_b.sum(), 1e-6)
            row  += f'  {dsc:>6.3f}'
        print(row)

## 7 · `hybrid_sample` — per-pixel DDPM/DDIM blending

At **every timestep** `t`:

1. Forward the current `x_t` through the UNet → `noise_pred`
2. Compute **DDPM** step → `x_ddpm` (stochastic, adds Gaussian noise)
3. Compute **DDIM** step → `x_ddim` (deterministic, no added noise)
4. Blend per-pixel: `x_{t-1} = mask * x_ddpm + (1 − mask) * x_ddim`

Inside the saliency mask the model must regenerate healthy anatomy from scratch
(stochastic).  Outside the mask the deterministic DDIM step preserves already-
healthy structure more accurately.

In [ ]:
@torch.no_grad()
def hybrid_sample(x_t: torch.Tensor,
                  mask: torch.Tensor,
                  encode_t: int    = T_ENCODE,
                  n_ddim_steps: int = N_DDIM_STEPS) -> torch.Tensor:
    """
    Hybrid DDPM (inside mask) / DDIM (outside mask) reverse diffusion.

    Parameters
    ----------
    x_t          : (1,1,256,256) noised image at timestep encode_t, float32
    mask         : (1,1,256,256) float32 binary saliency mask on DEVICE
    encode_t     : timestep to start denoising from
    n_ddim_steps : number of DDIM timestep intervals (higher = sharper/slower)

    Returns
    -------
    x_hat : (1,1,256,256) float32 healthy reconstruction
    """
    # build a sub-set of DDIM timesteps that span [encode_t, 0]
    # DDIMScheduler.set_timesteps picks uniformly-spaced steps over [0, T)
    # we then keep only those <= encode_t
    ddim_scheduler.set_timesteps(n_ddim_steps)
    timesteps = [int(t) for t in ddim_scheduler.timesteps if int(t) <= encode_t]
    if not timesteps:
        timesteps = list(range(encode_t, -1, -1))

    x_cur = x_t.to(DTYPE)
    mask_f = mask.float()

    for t_val in timesteps:
        t_batch = torch.full((x_cur.shape[0],), t_val, device=DEVICE, dtype=torch.long)

        # ── shared noise prediction ──────────────────────────────────────────
        noise_pred = unet(x_cur, t_batch).float()   # (1,1,256,256)
        x_cur_f    = x_cur.float()

        # ── DDIM step (deterministic) ────────────────────────────────────────
        x_ddim = ddim_scheduler.step(noise_pred, t_val, x_cur_f)[0]

        # ── DDPM step (stochastic) ───────────────────────────────────────────
        x_ddpm = ddpm_scheduler.step(noise_pred, t_val, x_cur_f)[0]

        # ── per-pixel blend ──────────────────────────────────────────────────
        # inside mask  → stochastic DDPM  (regenerate healthy anatomy)
        # outside mask → deterministic DDIM (preserve healthy structure)
        x_cur = (mask_f * x_ddpm + (1.0 - mask_f) * x_ddim).to(DTYPE)

    x_hat = ((x_cur.float() + 1.0) / 2.0).clamp(0.0, 1.0)
    return x_hat

## 8 · Pass 2 — saliency-guided hybrid inference

In [ ]:
@torch.no_grad()
def pass2_reconstruct(x0: torch.Tensor,
                      saliency_mask: torch.Tensor,
                      encode_t: int    = T_ENCODE,
                      n_ddim_steps: int = N_DDIM_STEPS) -> torch.Tensor:
    """
    Pass 2: add noise → hybrid_sample with saliency mask.

    Parameters
    ----------
    x0            : (1,1,256,256) float32, values in [0,1]
    saliency_mask : (1,1,256,256) float32 {0,1} from build_saliency_mask()
    encode_t      : noise level
    n_ddim_steps  : DDIM step count for the hybrid decoder

    Returns
    -------
    x_hat_clean : (1,1,256,256) float32 cleaned-up reconstruction
    """
    x0_scaled = x0 * 2.0 - 1.0
    noise     = torch.randn_like(x0_scaled)
    t_vec     = torch.tensor([encode_t], device=DEVICE).long()
    x_t       = ddpm_scheduler.add_noise(x0_scaled, noise, t_vec)

    x_hat_clean = hybrid_sample(
        x_t, saliency_mask,
        encode_t=encode_t,
        n_ddim_steps=n_ddim_steps,
    )
    return x_hat_clean

## 9 · Full two-pass pipeline wrapper

`run_inference()` respects the `USE_SALIENCY` flag.  Set it to `False` and
you get the original single-pass pipeline (Pass 1 only).

In [ ]:
def run_inference(x0: torch.Tensor,
                  use_saliency: bool = USE_SALIENCY,
                  encode_t: int = T_ENCODE,
                  n_ddim_steps: int = N_DDIM_STEPS,
                  thresh_method: str = THRESH_METHOD,
                  fixed_thresh: float = FIXED_THRESH,
                  smooth_sigma: float = SMOOTH_SIGMA):
    """
    Full inference pipeline for one slice.

    Parameters
    ----------
    x0           : (1,1,256,256) float32 on DEVICE, values in [0,1]
    use_saliency : True  → two-pass (Pass1 + Pass2)
                   False → single-pass DDPM only

    Returns dict with keys:
      'amap_rough'     : (256,256) float32  — Pass 1 anomaly map
      'saliency_mask'  : (256,256) float32  — binary mask (None if use_saliency=False)
      'thresh_val'     : float              — threshold used
      'amap_clean'     : (256,256) float32  — Pass 2 anomaly map (=amap_rough if no saliency)
      'x_hat_rough'    : (1,1,256,256) float32
      'x_hat_clean'    : (1,1,256,256) float32
    """
    # ── Pass 1: full-image DDPM reconstruction ────────────────────────────────
    x_hat_rough = ddpm_reconstruct(x0, encode_t=encode_t)
    amap_rough  = anomaly_map(x0, x_hat_rough, smooth_sigma=smooth_sigma)

    if not use_saliency:
        return dict(
            amap_rough   =amap_rough,
            saliency_mask=None,
            thresh_val   =None,
            amap_clean   =amap_rough,
            x_hat_rough  =x_hat_rough,
            x_hat_clean  =x_hat_rough,
        )

    # ── Build saliency mask from Pass 1 ──────────────────────────────────────
    mask_t, thresh_val = build_saliency_mask(
        amap_rough, method=thresh_method, fixed_val=fixed_thresh
    )

    # ── Pass 2: hybrid DDPM/DDIM with saliency mask ───────────────────────────
    x_hat_clean = pass2_reconstruct(
        x0, mask_t, encode_t=encode_t, n_ddim_steps=n_ddim_steps
    )
    amap_clean = anomaly_map(x0, x_hat_clean, smooth_sigma=smooth_sigma)

    return dict(
        amap_rough   =amap_rough,
        saliency_mask=mask_t[0, 0].cpu().numpy(),   # (256,256)
        thresh_val   =thresh_val,
        amap_clean   =amap_clean,
        x_hat_rough  =x_hat_rough,
        x_hat_clean  =x_hat_clean,
    )

## 10 · Single-slice demo

In [ ]:
# ── load one case, pick the most tumour-heavy slice ───────────────────────────
demo_case = all_cases[0]
slices0, z_ids0 = preprocess_volume(demo_case['t1_path'])
segs0           = preprocess_seg(demo_case['seg_path'], z_ids0)

tumor_load = [s.sum() for s in segs0]
best_z     = int(np.argmax(tumor_load))

x0_np  = slices0[best_z]   # (1,256,256)
seg_np = segs0[best_z]     # (1,256,256)

x0 = torch.from_numpy(x0_np)[None].to(DEVICE).float()   # (1,1,256,256)

print(f'Case : {demo_case["id"]}   z={z_ids0[best_z]}')
print(f'Tumour pixels: {(seg_np > 0).sum()}')

# ── show threshold comparison on this slice before running full inference ─────
# (uses a quick cheap anomaly map; replace with Pass-1 output after inference)
print('\nThreshold comparison table (will update after Pass 1 runs):')

In [ ]:
# ── run the full pipeline ─────────────────────────────────────────────────────
# This cell takes ~1-3 min on an RTX 5090 for T_ENCODE=500.
# Reduce T_ENCODE or increase N_DDIM_STEPS for faster/better quality trade-off.

result = run_inference(x0, use_saliency=USE_SALIENCY)

amap_rough    = result['amap_rough']
saliency_mask = result['saliency_mask']
thresh_val    = result['thresh_val']
amap_clean    = result['amap_clean']
x_hat_rough   = result['x_hat_rough']
x_hat_clean   = result['x_hat_clean']

print(f'Pass 1 anomaly map — min={amap_rough.min():.4f}  max={amap_rough.max():.4f}')
if USE_SALIENCY:
    pct_pos = 100.0 * saliency_mask.mean()
    print(f'Saliency mask thresh={thresh_val:.4f}  ({pct_pos:.1f}% positive pixels)')
    print(f'Pass 2 anomaly map — min={amap_clean.min():.4f}  max={amap_clean.max():.4f}')

In [ ]:
# ── threshold comparison table (now with Pass-1 amap) ────────────────────────
gt_binary = (seg_np[0] > 0).astype(np.float32)
compare_thresholds(amap_rough, gt_mask=gt_binary)

In [ ]:
# ── visualisation ─────────────────────────────────────────────────────────────
seg_plot = seg_np[0]                              # (256,256)
gt_cont  = (seg_plot > 0).astype(np.float32)

if USE_SALIENCY:
    fig, axes = plt.subplots(1, 7, figsize=(32, 5))
else:
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

ax = axes

# 0 — input
ax[0].imshow(x0[0,0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
ax[0].contour(gt_cont, levels=[0.5], colors='lime', linewidths=1)
ax[0].set_title(f'Input x₀\n{demo_case["id"]} z={z_ids0[best_z]}\n(green=GT tumour)', fontsize=8)

# 1 — Pass 1 reconstruction
ax[1].imshow(x_hat_rough[0,0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
ax[1].set_title(f'Pass 1 recon x̂₀ (DDPM)\nT_enc={T_ENCODE}', fontsize=8)

# 2 — Pass 1 anomaly map
im2 = ax[2].imshow(amap_rough, cmap='hot', vmin=0)
ax[2].contour(gt_cont, levels=[0.5], colors='cyan', linewidths=0.8)
ax[2].set_title('Pass 1 anomaly map\n|x₀ − x̂₀_rough|', fontsize=8)
fig.colorbar(im2, ax=ax[2], fraction=0.046)

# 3 — GT seg
ax[3].imshow(seg_plot, cmap='tab10', vmin=0, vmax=3)
ax[3].set_title('GT segmentation\n(0=bg 1=NCR 2=ED 3=ET)', fontsize=8)

if USE_SALIENCY:
    # 4 — saliency mask
    ax[4].imshow(saliency_mask, cmap='hot', vmin=0, vmax=1)
    ax[4].contour(gt_cont, levels=[0.5], colors='cyan', linewidths=0.8)
    ax[4].set_title(f'Saliency mask\n(thresh={thresh_val:.3f}, {THRESH_METHOD})', fontsize=8)

    # 5 — Pass 2 reconstruction
    ax[5].imshow(x_hat_clean[0,0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    ax[5].set_title('Pass 2 recon x̂₀ (hybrid)\nDDIM out / DDPM in mask', fontsize=8)

    # 6 — Pass 2 anomaly map
    im6 = ax[6].imshow(amap_clean, cmap='hot', vmin=0)
    ax[6].contour(gt_cont, levels=[0.5], colors='cyan', linewidths=0.8)
    ax[6].set_title('Pass 2 anomaly map\n|x₀ − x̂₀_clean|', fontsize=8)
    fig.colorbar(im6, ax=ax[6], fraction=0.046)

for a in ax:
    a.axis('off')

plt.tight_layout()
plt.savefig('saliency_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → saliency_demo.png')

## 11 · Batch evaluation — DSC, AUROC, pixel-AP

Runs both the rough (Pass 1 only) and clean (two-pass) pipelines on a set of
BraTS cases and reports slice-level metrics.

In [ ]:
def dice(pred_binary: np.ndarray, gt_binary: np.ndarray) -> float:
    inter = (pred_binary * gt_binary).sum()
    return float(2 * inter / max(pred_binary.sum() + gt_binary.sum(), 1e-6))


def best_f1_threshold(amap: np.ndarray, gt_binary: np.ndarray,
                      n_thresholds: int = 50) -> tuple[float, float]:
    """Sweep thresholds and return (best_DSC, best_threshold)."""
    lo, hi = amap.min(), amap.max()
    best_dsc, best_t = 0.0, lo
    for t in np.linspace(lo, hi, n_thresholds):
        d = dice((amap >= t).astype(np.float32), gt_binary)
        if d > best_dsc:
            best_dsc, best_t = d, float(t)
    return best_dsc, best_t


def evaluate_slice(amap: np.ndarray, gt_binary: np.ndarray):
    """Return dict of slice-level metrics."""
    metrics = {}
    flat_amap = amap.flatten()
    flat_gt   = gt_binary.flatten().astype(int)

    if flat_gt.sum() > 0 and flat_gt.sum() < flat_gt.size:
        metrics['auroc']  = roc_auc_score(flat_gt, flat_amap)
        metrics['pixel_ap'] = average_precision_score(flat_gt, flat_amap)
    else:
        metrics['auroc']    = float('nan')
        metrics['pixel_ap'] = float('nan')

    metrics['best_dsc'], metrics['best_thresh'] = best_f1_threshold(amap, gt_binary)

    # DSC at the global Otsu threshold
    try:
        otsu_t = threshold_otsu(amap)
    except Exception:
        otsu_t = 0.05
    metrics['dsc_otsu'] = dice((amap >= otsu_t).astype(np.float32), gt_binary)
    return metrics

In [ ]:
# ── batch eval config ─────────────────────────────────────────────────────────
N_EVAL_CASES   = 5       # set to None for all cases (slow!)
TUMOR_ONLY     = True    # only evaluate slices that contain tumour

eval_cases = all_cases[:N_EVAL_CASES] if N_EVAL_CASES else all_cases
eval_ds    = BraTSDataset(eval_cases, tumor_only=TUMOR_ONLY)
eval_dl    = DataLoader(eval_ds, batch_size=1, shuffle=False,
                        num_workers=0, collate_fn=brats_collate)

print(f'Evaluating on {len(eval_ds)} tumor slices from {len(eval_cases)} cases')

In [ ]:
rows_rough, rows_clean = [], []

for i, batch in enumerate(eval_dl):
    x0_b  = batch['image'].to(DEVICE).float()    # (1,1,256,256)
    gt_b  = batch['seg'][0, 0]                   # (256,256) int32
    gt_bi = (gt_b > 0).astype(np.float32)

    res = run_inference(x0_b, use_saliency=USE_SALIENCY)

    m_rough = evaluate_slice(res['amap_rough'], gt_bi)
    m_clean = evaluate_slice(res['amap_clean'], gt_bi)

    rows_rough.append(m_rough)
    rows_clean.append(m_clean)

    tag = '[saliency]' if USE_SALIENCY else '[single-pass]'
    print(f'  [{i+1}/{len(eval_dl)}] {batch["case"][0]} z={batch["z_idx"][0]}  '
          f'rough AUROC={m_rough["auroc"]:.3f}  '
          f'clean AUROC={m_clean["auroc"]:.3f}  {tag}')

print('Done.')

In [ ]:
# ── summary table ─────────────────────────────────────────────────────────────
def nanmean(vals):
    v = [x for x in vals if not np.isnan(x)]
    return np.mean(v) if v else float('nan')

metrics_to_show = ['auroc', 'pixel_ap', 'best_dsc', 'dsc_otsu']

print(f'\n{'Metric':>14}  {'Rough (P1)':>12}  {'Clean (P2)':>12}  {'Delta':>8}')
print('-' * 52)
for m in metrics_to_show:
    r = nanmean([row[m] for row in rows_rough])
    c = nanmean([row[m] for row in rows_clean])
    delta = c - r
    print(f'{m:>14}  {r:>12.4f}  {c:>12.4f}  {delta:>+8.4f}')